# Archiving deepdive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AiMTT-project/UC7-SAIL/blob/main/3.4%20Evaluation%20during%20the%20event/Assignment-Solution/archiving_deepdive_answers.ipynb)


### Introduction to archiving

Real-time data analysis and live insights are highly valuable during events, especially for supporting operational decisions in dynamic crowd situations. However, analysis during the *cold* or *lukewarm* phase can be even more useful. Looking back at historical data makes it possible to evaluate patterns, understand what happened, and improve planning and decision-making for future events.

To make this possible, data from live operations must be archived efficiently during the event itself. Without a well-structured archive, valuable information may be lost or become too difficult to use afterward. Good data archiving therefore forms the foundation for meaningful post-event analysis by both data scientists and crowd managers.

### Purpose of this Notebook

This notebook demonstrates how event data can be archived and structured in a way that enables effective post-event analysis. It focuses on practical approaches that support collaboration between data scientists and crowd managers.

### Case Study: SAIL 2025

The examples in this notebook are based on data from **SAIL 2025**. This case study illustrates how archived data can be explored to extract insights, evaluate crowd dynamics, and support better preparation for future large-scale events.

### Key Takeaways

- Real-time insights are important, but post-event analysis often provides deeper understanding  
- Efficient data archiving during events is essential for later use  
- Structured historical data enables better evaluation and planning  
- Collaboration between data scientists and crowd managers benefits from accessible data workflows  

### Import necessary libraries


In [ ]:
!pip install geopandas pyarrow fastparquet

In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd
import geopandas as gpd


### Define functions


In [ ]:

def reset_output_folder(folder_path):
    """Delete an output folder if it already exists and recreate it."""
    folder = Path(folder_path)
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def show_directory_tree(folder_path):
    """Print a small directory tree to inspect the written archive."""
    folder = Path(folder_path)
    for root, dirs, files in os.walk(folder):
        level = root.replace(str(folder), "").count(os.sep)
        indent = "    " * level
        print(f"{indent}{Path(root).name}/")
        subindent = "    " * (level + 1)
        for file in files:
            print(f"{subindent}{file}")


### Retrieve necessary data

This module uses `LOS_archive_data.geojson`, hosted in the `2.2 LOS` folder on GitHub.

- **Local execution**: The notebook checks local and repository directories.
- **Google Colab**: The code cell below automatically downloads the file from GitHub into `sample_data/`.
- **Optional manual Colab command**:
  ```bash
  !mkdir -p sample_data
  !wget -q https://raw.githubusercontent.com/AiMTT-project/UC7-SAIL/main/2.2%20LOS/LOS_archive_data.geojson -O sample_data/LOS_archive_data.geojson
  ```


In [ ]:
import os
import urllib.request

# Dataset path configuration (compatible with local environments and Google Colab)
data_filename = "LOS_archive_data.geojson"
data_url = "https://raw.githubusercontent.com/AiMTT-project/UC7-SAIL/main/2.2%20LOS/LOS_archive_data.geojson"

candidate_paths = [
    data_filename,
    os.path.join("..", data_filename),
    os.path.join("..", "..", "2.2 LOS", data_filename),
    os.path.join("sample_data", data_filename),
    os.path.join("/content/sample_data", data_filename),
]

archive_data = None
for p in candidate_paths:
    if os.path.exists(p):
        archive_data = p
        break

if archive_data is None:
    os.makedirs("sample_data", exist_ok=True)
    archive_data = os.path.join("sample_data", data_filename)
    print(f"Downloading {data_filename} from GitHub...")
    urllib.request.urlretrieve(data_url, archive_data)
    print(f"Saved to {archive_data}")

print(f"Using dataset: {archive_data}")


In [ ]:
archive_gdf = gpd.read_file(archive_data)
archive_gdf["time"] = pd.to_datetime(archive_gdf["time"])
archive_gdf = archive_gdf.sort_values(["location", "time"]).reset_index(drop=True)


### Understanding the data


First off, we are going to perform some basic exploration of the data we'll be working with. To start, lets explore which attributes the dataset has.


In [ ]:
df_gdf_info = pd.DataFrame({
    "Attribute": archive_gdf.columns,
    "Data type": archive_gdf.dtypes.astype(str).values
})

df_gdf_info


,Attribute,Data type
0,sensor,object
1,time,"datetime64[ms, UTC+02:00]"
2,count_line,int32
3,location,object
4,count_area,float64
5,countline_length,int32
6,geometry,geometry


In [ ]:
archive_gdf.head(2)


,sensor,time,count_line,location,count_area,countline_length,geometry
0,GASA-02-01,2025-08-20 08:10:00+02:00,-1,BrugOostertoegang1,-1.0,6,"POLYGON ((122181.32 487968.791, 122179.408 487..."
1,GASA-02-01,2025-08-20 08:11:00+02:00,3,BrugOostertoegang1,4.0,6,"POLYGON ((122181.32 487968.791, 122179.408 487..."


So we are working with a dataset which tracks counts for multiple sensors and locations over a period of time. Lets see what we can find out about the time granularity and extent of the geodataframe.


In [ ]:
start_time = archive_gdf['time'].min()
end_time = archive_gdf['time'].max()
time_diffs_per_loc = (
    archive_gdf
    .sort_values(['location', 'time'])
    .groupby('location')['time']
    .diff()
    .dropna()
)


print("Start time:", start_time)
print("End time:", end_time)
print("Number of records:", len(archive_gdf))
print("Number of locations:", archive_gdf['location'].nunique())
print("Median time step:", time_diffs_per_loc.median())


Start time: 2025-08-20 08:09:00+02:00
End time: 2025-08-20 16:28:00+02:00
Number of records: 1577
Number of locations: 5
Median time step: 0 days 00:00:00


This shows that we are working with LOS-related pedestrian data from 20 August 2025 between the morning and late afternoon. Because the dataset contains repeated measurements over time for multiple locations, it is a good example of data that benefits from structured archiving.


In [ ]:
archive_gdf[['location', 'time']].groupby('location').agg(['min', 'max', 'count'])

time                                
                                         min                       max count
location                                                                    
BrugOostertoegang1 2025-08-20 08:10:00+02:00 2025-08-20 16:28:00+02:00   266
BrugOostertoegang2 2025-08-20 08:10:00+02:00 2025-08-20 16:28:00+02:00   323
CSriOosterdokskade 2025-08-20 08:10:00+02:00 2025-08-20 16:28:00+02:00   323
IJBoulevard        2025-08-20 08:09:00+02:00 2025-08-20 16:28:00+02:00   342
Oosterdoksbrug     2025-08-20 08:10:00+02:00 2025-08-20 16:28:00+02:00   323

### Archiving Data Efficiently with Parquet and Compression

A key step in building reliable data pipelines is archiving data in a format that is both efficient and easy to reuse. Instead of storing raw files (such as GeoJSON), it is common to convert data into a columnar storage format like Parquet. This format is designed for analytical workloads and offers significant advantages in terms of storage size and read performance.

One of the main benefits of Parquet is compression. Parquet applies efficient encoding and compression techniques (such as Snappy or Gzip) at the column level. This means that repetitive or similar values within a column are stored much more compactly than in row-based formats. As a result, archived datasets can be several times smaller while still being fast to read.

Another advantage is that Parquet preserves data types and schema, making it easier for downstream users to work with the data consistently. In the case of geospatial data, using GeoParquet ensures that geometry columns are stored in a standardized way, enabling interoperability with tools like GeoPandas.

In this exercise, we focus on converting and archiving the input GeoJSON data into GeoParquet format. By iterating over the dataset in a loop, we create structured, compressed output files that are ready for efficient storage and later analysis


*Assignment 1:* Archive the dataset to Parquet using a loop over the data.

In [ ]:
import os
import urllib.request

# Dataset path configuration (compatible with local environments and Google Colab)
data_filename = "LOS_archive_data.geojson"
data_url = "https://raw.githubusercontent.com/AiMTT-project/UC7-SAIL/main/2.2%20LOS/LOS_archive_data.geojson"

candidate_paths = [
    data_filename,
    os.path.join("..", data_filename),
    os.path.join("..", "..", "2.2 LOS", data_filename),
    os.path.join("sample_data", data_filename),
    os.path.join("/content/sample_data", data_filename),
]

archive_data = None
for p in candidate_paths:
    if os.path.exists(p):
        archive_data = p
        break

if archive_data is None:
    os.makedirs("sample_data", exist_ok=True)
    archive_data = os.path.join("sample_data", data_filename)
    print(f"Downloading {data_filename} from GitHub...")
    urllib.request.urlretrieve(data_url, archive_data)
    print(f"Saved to {archive_data}")

print(f"Using dataset: {archive_data}")


archiving/
    stream_archive.parquet
Number of parquet files written: 1


In [ ]:
example_gdf_assignment_1 = gpd.read_parquet(archived_files_assignment_1[0])
example_gdf_assignment_1.head(2)


,sensor,time,count_line,location,count_area,countline_length,geometry
0,GASA-02-01,2025-08-20 08:10:00+02:00,-1,BrugOostertoegang1,-1.0,6,"POLYGON ((122181.32038 487968.7915, 122179.408..."
1,GASA-02-01,2025-08-20 08:11:00+02:00,3,BrugOostertoegang1,4.0,6,"POLYGON ((122181.32038 487968.7915, 122179.408..."


The archive above is easy to inspect and easy to share. However, when the archive grows, writing files without temporal partitioning can become less efficient. If an analyst only needs data for one specific hour, they may still have to open larger files than necessary.


### Organising Archived Data by Time Using Partitioning

When working with larger datasets, it becomes important not just to store data efficiently, but also to organise it in a way that makes it easy to retrieve. A widely used strategy for this is partitioning, where data is physically divided into separate folders based on a specific attribute such as date, hour, or location.

In this exercise, we organise the archive by hour. This means that all records belonging to the same hour are stored together in the same folder. Using a loop makes this process explicit: for each hour in the dataset, we filter the relevant records and write them to a dedicated location.

Partitioning offers several important benefits:

- Faster data access: When querying data, only the relevant partitions need to be read. For example, if you are interested in a specific hour, you can skip all other data.
- Reduced I/O and compute cost: Reading smaller subsets of data reduces disk usage and speeds up processing, which is especially important for large-scale systems.
- Scalability: As datasets grow, partitioning helps maintain performance by preventing single large files from becoming bottlenecks.
- Better organisation: The folder structure itself becomes meaningful, making it easier to understand and navigate the archive.
- Compatibility with data tools: Many data processing frameworks (such as Spark or DuckDB) automatically recognise partitioned structures and optimise queries accordingly.

This kind of setup is particularly useful for time-based data, such as event logs or sensor measurements, where analysis is often focused on specific time windows (e.g. peak hours or incidents).

*Assignment 2:* Archive streaming JSON records to Parquet and partition the archive by hour.


In [ ]:
archive_gdf.head(1)

,sensor,time,count_line,location,count_area,countline_length,geometry
0,GASA-02-01,2025-08-20 08:10:00+02:00,-1,BrugOostertoegang1,-1.0,6,"POLYGON ((122181.32 487968.791, 122179.408 487..."


In [ ]:
from pathlib import Path
assignment_2_folder ="/content/sample_data/archiving/assignment_2"
output_folder_assignment_2 = reset_output_folder(assignment_2_folder)


def archive_record_by_hour(record_json, output_folder):
    """
    Archive one incoming JSON record into the correct hour partition.

    Students:
    - read the JSON-like dictionary
    - extract the hour from the timestamp
    - create an hour-based folder
    - convert the record into a one-row GeoDataFrame
    - append it to the Parquet archive for that hour
    """
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt

    # Read the timestamp and extract the hour
    timestamp = pd.to_datetime(record_json["time"])
    hour_value = timestamp.hour

    # Use a normal folder name instead of hive-style "hour=08"
    partition_folder = Path(output_folder) / f"hour_{hour_value:02d}"
    partition_folder.mkdir(parents=True, exist_ok=True)

    # Define the parquet file inside the hour partition
    output_file = partition_folder / "data.parquet"

    # Convert the JSON-like dictionary to a one-row DataFrame
    record_df = pd.DataFrame([record_json])

    # Convert the geometry from WKT text back to a shapely geometry
    record_df["geometry"] = record_df["geometry"].apply(wkt.loads)

    # Turn the DataFrame into a GeoDataFrame
    record_gdf = gpd.GeoDataFrame(record_df, geometry="geometry", crs="EPSG:4326")

    # If the partition archive already exists, append the new record
    if output_file.exists():
        existing_gdf = gpd.read_parquet(output_file)
        combined_df = pd.concat([existing_gdf, record_gdf], ignore_index=True)
        record_gdf = gpd.GeoDataFrame(combined_df, geometry="geometry", crs=existing_gdf.crs)

    # Write the updated partition to Parquet
    record_gdf.to_parquet(output_file, index=False)


# Simulate a data stream: records pass by one by one as JSON-like dictionaries
for _, row in archive_gdf.iterrows():
    incoming_record = row.drop(labels="geometry").to_dict()
    incoming_record["geometry"] = row.geometry.wkt

    # Archive the record while the stream is "running"
    archive_record_by_hour(incoming_record, output_folder_assignment_2)


show_directory_tree(output_folder_assignment_2)

archived_files_assignment_2 = sorted(output_folder_assignment_2.glob("hour_*/*.parquet"))
print("Number of parquet files written:", len(archived_files_assignment_2))

archiving_hour/
    hour_08/
        data.parquet
    hour_09/
        data.parquet
    hour_10/
        data.parquet
    hour_11/
        data.parquet
    hour_12/
        data.parquet
    hour_13/
        data.parquet
    hour_14/
        data.parquet
    hour_15/
        data.parquet
    hour_16/
        data.parquet
Number of parquet files written: 9


In [ ]:
example_gdf_assignment_2 = gpd.read_parquet(archived_files_assignment_2[0])
example_gdf_assignment_2.head(2)

,sensor,time,count_line,location,count_area,countline_length,geometry
0,GASA-02-01,2025-08-20 08:10:00+02:00,-1,BrugOostertoegang1,-1.0,6,"POLYGON ((122181.32038 487968.7915, 122179.408..."
1,GASA-02-01,2025-08-20 08:11:00+02:00,3,BrugOostertoegang1,4.0,6,"POLYGON ((122181.32038 487968.7915, 122179.408..."


## Conclusion

In this notebook, we explored two practical ways to archive geospatial event data to Parquet.

First, we created a simple non-partitioned archive, where the dataset was written to separate Parquet files per location. This structure is easy to understand and works well for relatively small archives.

Next, we created an hour-partitioned archive. By introducing folders per hour, the archive becomes more scalable and more efficient for later analysis, because users can directly target the relevant time window.

The key takeaway is that archiving is not only about saving data, but also about organising it in a way that supports future analysis. Choosing between a simple export and a partitioned structure depends on the expected archive size, the type of queries that will be performed later, and the operational context in which the data will be reused.
